# Computational Notebook 16: Energy & Sustainability

## Overview

Blockchain energy consumption -- particularly Bitcoin's Proof of Work -- is one of the most debated topics in the cryptocurrency space. This notebook quantifies the environmental impact of blockchain networks from first principles: estimating energy consumption from hashrate data, calculating carbon footprints by region, analyzing the Ethereum Merge's ~99.95% energy reduction, modeling mining geography and renewable energy adoption, assessing e-waste from ASIC (Application-Specific Integrated Circuit) hardware lifecycles, and projecting future sustainability scenarios.

## Prerequisites
- **Notebook 07**: Mining Economics (hashrate, difficulty, block rewards)
- **Notebook 14**: Consensus Simulations (PoW vs PoS mechanics)
- Basic Python programming and familiarity with NumPy

## Learning Objectives

1. Estimate Bitcoin's energy consumption from hashrate using CBECI methodology
2. Calculate carbon footprints using regional energy mix and emission factors
3. Quantify the Ethereum Merge's energy reduction from PoW to PoS
4. Analyze mining geography, renewable energy adoption, and stranded energy utilization
5. Model ASIC hardware e-waste and environmental lifecycle impacts
6. Apply ESG (Environmental, Social, Governance) scoring frameworks to crypto protocols
7. Project future energy consumption under different technology and adoption scenarios

**Estimated Time:** 4-6 hours

**Related Content:** [Section 09: Sustainability](../sections/09-sustainability.md)

In [ ]:
# Setup and imports
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field
from collections import defaultdict

# Plot settings
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True

print("All imports successful!")
print("This notebook analyzes blockchain energy consumption and sustainability.")

---
## 1. Energy Consumption Modeling

Bitcoin's energy consumption can be estimated from its network hashrate and the efficiency of mining hardware.

> **Definition: CBECI (Cambridge Bitcoin Electricity Consumption Index)** -- A methodology developed by the Cambridge Centre for Alternative Finance that estimates Bitcoin's energy consumption by modeling the total hashrate and the efficiency range of mining hardware in operation.

The basic formula:

$$\text{Power (W)} = \frac{\text{Hashrate (H/s)}}{\text{Efficiency (H/J)}}$$

$$\text{Annual Energy (TWh)} = \frac{\text{Power (W)} \times 8760}{10^{12}}$$

The CBECI provides bounds:
- **Lower bound**: Most efficient hardware only
- **Best estimate**: Weighted average of hardware in profitable operation
- **Upper bound**: Least efficient hardware still profitable

**Source:** Cambridge Centre for Alternative Finance. (2024). "Cambridge Bitcoin Electricity Consumption Index." cbeci.org

In [ ]:
@dataclass
class MiningHardware:
    """ASIC mining hardware specification."""
    model: str
    manufacturer: str
    hashrate_th: float     # Terahashes per second
    power_watts: float     # Power consumption in watts
    efficiency_jth: float  # Joules per terahash
    release_year: int
    price_usd: float


# ASIC hardware database (synthetic but based on real models)
hardware_db = [
    MiningHardware("S9", "Bitmain", 14, 1350, 96.4, 2017, 200),
    MiningHardware("S17", "Bitmain", 56, 2520, 45.0, 2019, 1500),
    MiningHardware("S19", "Bitmain", 95, 3250, 34.2, 2020, 3000),
    MiningHardware("S19 Pro", "Bitmain", 110, 3250, 29.5, 2020, 4000),
    MiningHardware("S19 XP", "Bitmain", 140, 3010, 21.5, 2022, 5000),
    MiningHardware("S21", "Bitmain", 200, 3550, 17.5, 2024, 6000),
    MiningHardware("M50S", "MicroBT", 126, 3276, 26.0, 2022, 4500),
    MiningHardware("M60S", "MicroBT", 186, 3441, 18.5, 2024, 5500),
]


def estimate_bitcoin_energy(network_hashrate_eh: float,
                             hardware_mix: Dict[str, float] = None) -> Dict[str, float]:
    """Estimate Bitcoin energy consumption from hashrate.
    
    Args:
        network_hashrate_eh: Network hashrate in exahashes/second
        hardware_mix: {model_name: fraction} of hardware in operation
    
    Returns:
        Dict with power (GW) and energy (TWh/year) estimates
    """
    hashrate_th = network_hashrate_eh * 1e6  # Convert EH/s to TH/s
    
    # If no mix specified, use efficiency bounds
    efficiencies = [h.efficiency_jth for h in hardware_db]
    
    best_eff = min(efficiencies)    # Most efficient
    worst_eff = max(efficiencies)   # Least efficient
    avg_eff = np.mean(efficiencies) # Average
    
    if hardware_mix:
        hw_dict = {h.model: h for h in hardware_db}
        weighted_eff = sum(hw_dict[m].efficiency_jth * frac
                         for m, frac in hardware_mix.items())
    else:
        weighted_eff = avg_eff
    
    def calc_power_and_energy(efficiency):
        power_w = hashrate_th * efficiency * 1e12 / 1e12  # TH/s * J/TH = W
        power_gw = power_w * efficiency / 1e9
        # Correct: power = hashrate * efficiency
        total_power_gw = hashrate_th * efficiency / 1e9
        energy_twh = total_power_gw * 8760 / 1000
        return total_power_gw, energy_twh
    
    lower_gw, lower_twh = calc_power_and_energy(best_eff)
    upper_gw, upper_twh = calc_power_and_energy(worst_eff)
    best_gw, best_twh = calc_power_and_energy(weighted_eff)
    
    return {
        'hashrate_eh': network_hashrate_eh,
        'lower_bound_gw': lower_gw,
        'upper_bound_gw': upper_gw,
        'best_estimate_gw': best_gw,
        'lower_bound_twh': lower_twh,
        'upper_bound_twh': upper_twh,
        'best_estimate_twh': best_twh,
        'weighted_efficiency': weighted_eff
    }


# Current Bitcoin network
current_hashrate = 600  # ~600 EH/s

print("=" * 60)
print("BITCOIN ENERGY CONSUMPTION ESTIMATION")
print("=" * 60)

print(f"\nASIC Hardware Database:")
print(f"{'Model':<14} {'Year':>6} {'TH/s':>8} {'Watts':>8} {'J/TH':>8}")
print("-" * 48)
for h in hardware_db:
    print(f"{h.model:<14} {h.release_year:>6} {h.hashrate_th:>8.0f} {h.power_watts:>8,.0f} {h.efficiency_jth:>7.1f}")

result = estimate_bitcoin_energy(current_hashrate)
print(f"\nNetwork hashrate: {current_hashrate} EH/s")
print(f"\n{'Estimate':<16} {'Power (GW)':>12} {'Energy (TWh/yr)':>16}")
print("-" * 48)
print(f"{'Lower bound':<16} {result['lower_bound_gw']:>11.1f} {result['lower_bound_twh']:>15.1f}")
print(f"{'Best estimate':<16} {result['best_estimate_gw']:>11.1f} {result['best_estimate_twh']:>15.1f}")
print(f"{'Upper bound':<16} {result['upper_bound_gw']:>11.1f} {result['upper_bound_twh']:>15.1f}")

In [ ]:
# Historical energy consumption and comparisons
np.random.seed(42)

# Synthetic historical hashrate (monthly, 6 years)
months = 72
time_months = np.arange(months)
hashrate_history = 50 * np.exp(0.04 * time_months) * (1 + 0.1 * np.sin(time_months / 6))
hashrate_history = np.minimum(hashrate_history, 700)  # Cap at current levels

# Efficiency improves over time (newer hardware)
efficiency_history = 60 * np.exp(-0.015 * time_months) + 15

# Calculate energy consumption over time
power_history = hashrate_history * 1e6 * efficiency_history / 1e9  # GW
energy_history = power_history * 8760 / 1000  # TWh/yr (annualized)

# Country comparison data
country_energy = {
    'Norway': 124, 'Argentina': 132, 'Bitcoin (est)': result['best_estimate_twh'],
    'Netherlands': 110, 'Sweden': 131, 'Switzerland': 56,
    'Google': 18.3, 'Banking System': 260,
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Energy over time
axes[0].plot(time_months, energy_history, 'r-', linewidth=2, label='BTC Energy (TWh/yr)')
ax_hash = axes[0].twinx()
ax_hash.plot(time_months, hashrate_history, 'b--', linewidth=1.5, alpha=0.5, label='Hashrate (EH/s)')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Energy (TWh/year)', color='red')
ax_hash.set_ylabel('Hashrate (EH/s)', color='blue')
axes[0].set_title('Bitcoin Energy Consumption Over Time')
axes[0].legend(loc='upper left')
ax_hash.legend(loc='center right')

# Right: Country comparison
sorted_countries = sorted(country_energy.items(), key=lambda x: x[1], reverse=True)
c_names = [c[0] for c in sorted_countries]
c_values = [c[1] for c in sorted_countries]
c_colors = ['orange' if 'Bitcoin' in n else 'lightblue' if n in ['Google', 'Banking System'] else 'steelblue'
            for n in c_names]

axes[1].barh(c_names, c_values, color=c_colors)
axes[1].set_xlabel('Annual Energy (TWh)')
axes[1].set_title('Bitcoin vs Country & Industry Energy Use')

plt.tight_layout()
plt.savefig('/tmp/btc_energy.png', dpi=100, bbox_inches='tight')
plt.show()
print("Bitcoin consumes energy comparable to mid-sized countries.")

---
## 2. Carbon Footprint Analysis

Energy consumption alone doesn't determine environmental impact -- the carbon intensity depends on the energy source mix.

> **Definition: Carbon Intensity** -- The amount of CO2 emitted per unit of energy produced, measured in grams of CO2 per kilowatt-hour (gCO2/kWh). Coal: ~900 gCO2/kWh, Natural Gas: ~450, Solar: ~40, Nuclear: ~12, Hydro: ~4.

$$\text{CO}_2\text{ (tonnes)} = \text{Energy (kWh)} \times \text{Carbon Intensity (gCO}_2\text{/kWh)} / 10^6$$

In [ ]:
# Energy source carbon intensities (gCO2/kWh)
carbon_intensity = {
    'Coal': 900, 'Natural Gas': 450, 'Oil': 700,
    'Nuclear': 12, 'Hydro': 4, 'Wind': 11,
    'Solar': 40, 'Geothermal': 38, 'Biomass': 230,
}

# Mining energy mix by region
regional_mix = {
    'USA (Texas)': {'Natural Gas': 0.45, 'Wind': 0.25, 'Coal': 0.15, 'Nuclear': 0.10, 'Solar': 0.05},
    'USA (NY)': {'Hydro': 0.40, 'Natural Gas': 0.30, 'Nuclear': 0.20, 'Wind': 0.10},
    'Kazakhstan': {'Coal': 0.70, 'Natural Gas': 0.20, 'Hydro': 0.10},
    'Russia': {'Natural Gas': 0.50, 'Hydro': 0.20, 'Coal': 0.15, 'Nuclear': 0.15},
    'Canada': {'Hydro': 0.60, 'Nuclear': 0.15, 'Natural Gas': 0.15, 'Wind': 0.10},
    'Iceland': {'Geothermal': 0.65, 'Hydro': 0.35},
    'Norway': {'Hydro': 0.90, 'Wind': 0.10},
}

# Mining distribution by region
mining_share = {
    'USA (Texas)': 0.25, 'USA (NY)': 0.10, 'Kazakhstan': 0.10,
    'Russia': 0.12, 'Canada': 0.10, 'Iceland': 0.03, 'Norway': 0.02,
}
# Remaining 28% is "Other" with world average mix

def weighted_carbon_intensity(energy_mix: Dict[str, float]) -> float:
    """Calculate weighted carbon intensity for an energy mix."""
    return sum(carbon_intensity[source] * fraction
              for source, fraction in energy_mix.items())


print("=" * 60)
print("CARBON FOOTPRINT ANALYSIS")
print("=" * 60)

# Carbon intensity by region
print(f"\n{'Region':<18} {'Carbon Intensity':>18} {'Mining Share':>13} {'Renewable %':>13}")
print("-" * 65)

weighted_total = 0
for region, mix in regional_mix.items():
    ci = weighted_carbon_intensity(mix)
    share = mining_share.get(region, 0)
    renewable_sources = {'Hydro', 'Wind', 'Solar', 'Geothermal'}
    renewable_pct = sum(v for k, v in mix.items() if k in renewable_sources) * 100
    weighted_total += ci * share
    print(f"{region:<18} {ci:>14.0f} gCO2/kWh {share:>11.0%} {renewable_pct:>11.0f}%")

# Network-wide estimate
btc_energy_kwh = result['best_estimate_twh'] * 1e9  # Convert TWh to kWh
# Approximate remaining 28% with world average
world_avg_ci = 475  # gCO2/kWh world average
remaining_share = 1 - sum(mining_share.values())
network_ci = weighted_total + world_avg_ci * remaining_share

co2_tonnes = btc_energy_kwh * network_ci / 1e6
co2_mt = co2_tonnes / 1e6

print(f"\nNetwork-wide weighted carbon intensity: {network_ci:.0f} gCO2/kWh")
print(f"Estimated annual CO2 emissions: {co2_mt:.1f} Mt CO2")
print(f"\nComparison: New Zealand emits ~34 Mt CO2/year")

In [ ]:
# Visualize carbon analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Carbon intensity by region
regions = list(regional_mix.keys())
intensities = [weighted_carbon_intensity(regional_mix[r]) for r in regions]
renewables = [sum(v for k, v in regional_mix[r].items()
                  if k in {'Hydro', 'Wind', 'Solar', 'Geothermal'}) * 100
              for r in regions]

colors = plt.cm.RdYlGn_r(np.array(intensities) / max(intensities))
bars = axes[0].barh(regions, intensities, color=colors)
axes[0].set_xlabel('Carbon Intensity (gCO2/kWh)')
axes[0].set_title('Carbon Intensity by Mining Region')

# Right: CO2 per transaction comparison across chains
chain_co2 = {
    'Bitcoin': co2_mt * 1e6 / (0.3e6 * 365),  # tonnes per tx
    'Ethereum (PoW era)': 0.035,  # Historical estimate
    'Ethereum (PoS)': 0.000003,
    'Solana': 0.000001,
    'Visa': 0.0000004,
}

c_names = list(chain_co2.keys())
c_values = list(chain_co2.values())
c_colors = ['orange', 'lightblue', 'blue', 'purple', 'green']

axes[1].barh(c_names, [v * 1000 for v in c_values], color=c_colors)  # kg CO2
axes[1].set_xlabel('kg CO2 per Transaction')
axes[1].set_title('Carbon Footprint per Transaction')
axes[1].set_xscale('log')

plt.tight_layout()
plt.savefig('/tmp/carbon_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print("Carbon footprint varies enormously by region and consensus mechanism.")

---
## 3. The Ethereum Merge: PoW to PoS

On September 15, 2022, Ethereum transitioned from Proof of Work to Proof of Stake in an event called "The Merge." This reduced Ethereum's energy consumption by approximately 99.95%.

> **Definition: The Merge** -- Ethereum's transition from Proof of Work (energy-intensive mining) to Proof of Stake (validator staking) on September 15, 2022. The Beacon Chain (PoS) merged with the execution layer, eliminating mining entirely.

**Source:** Ethereum Foundation. (2022). "The Merge." ethereum.org/en/roadmap/merge/

In [ ]:
# Ethereum Merge energy analysis

# Pre-Merge (PoW) estimates
eth_pow_hashrate_th = 900e6  # ~900 TH/s (GPU mining)
eth_pow_efficiency = 0.4     # ~0.4 kWh per TH (GPU)
eth_pow_power_gw = eth_pow_hashrate_th * eth_pow_efficiency / 1e9
eth_pow_twh = eth_pow_power_gw * 8760 / 1000

# Post-Merge (PoS) estimates
n_validators = 900_000
power_per_validator_w = 10  # ~10W per validator node
eth_pos_power_gw = n_validators * power_per_validator_w / 1e9
eth_pos_twh = eth_pos_power_gw * 8760 / 1000

reduction = (1 - eth_pos_twh / eth_pow_twh) * 100

print("=" * 60)
print("ETHEREUM MERGE: ENERGY IMPACT")
print("=" * 60)

print(f"\n{'Metric':<30} {'Pre-Merge (PoW)':>18} {'Post-Merge (PoS)':>18}")
print("-" * 68)
print(f"{'Consensus mechanism':<30} {'Proof of Work':>18} {'Proof of Stake':>18}")
print(f"{'Network participants':<30} {'~1M GPUs':>18} {f'{n_validators:,} validators':>18}")
print(f"{'Power per unit':<30} {'~200W per GPU':>18} {'~10W per node':>18}")
print(f"{'Total power (GW)':<30} {eth_pow_power_gw:>17.3f} {eth_pos_power_gw:>17.6f}")
print(f"{'Annual energy (TWh)':<30} {eth_pow_twh:>17.2f} {eth_pos_twh:>17.5f}")
print(f"{'Energy reduction':<30} {'':>18} {reduction:>16.2f}%")

# CO2 savings
eth_co2_saved_mt = (eth_pow_twh - eth_pos_twh) * 1e9 * network_ci / 1e12
print(f"\nAnnual CO2 savings: {eth_co2_saved_mt:.1f} Mt CO2")
print(f"Equivalent to removing ~{eth_co2_saved_mt / 0.0046:.0f} cars from the road")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar comparison
categories = ['Power (MW)', 'Energy (GWh/yr)', 'CO2 (kt/yr)']
pow_vals = [eth_pow_power_gw * 1000, eth_pow_twh * 1000, eth_pow_twh * 1e9 * network_ci / 1e9]
pos_vals = [eth_pos_power_gw * 1000, eth_pos_twh * 1000, eth_pos_twh * 1e9 * network_ci / 1e9]

x = np.arange(len(categories))
width = 0.35
axes[0].bar(x - width/2, pow_vals, width, label='Pre-Merge (PoW)', color='red', alpha=0.7)
axes[0].bar(x + width/2, pos_vals, width, label='Post-Merge (PoS)', color='green', alpha=0.7)
axes[0].set_xticks(x)
axes[0].set_xticklabels(categories)
axes[0].set_ylabel('Value')
axes[0].set_title('Ethereum Energy: PoW vs PoS')
axes[0].set_yscale('log')
axes[0].legend()

# Timeline
merge_month = 36  # Month 36 = Merge
months_timeline = np.arange(72)
eth_energy_timeline = np.where(months_timeline < merge_month,
                                eth_pow_twh * (0.8 + 0.2 * months_timeline / merge_month),
                                eth_pos_twh)

axes[1].plot(months_timeline, eth_energy_timeline, 'b-', linewidth=2)
axes[1].axvline(x=merge_month, color='red', linestyle='--', linewidth=2, label='The Merge')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Energy (TWh/yr)')
axes[1].set_title('Ethereum Energy Consumption Timeline')
axes[1].legend()
axes[1].set_yscale('log')
axes[1].annotate('99.95% reduction', xy=(merge_month + 2, 0.001),
                fontsize=12, color='green', fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/ethereum_merge.png', dpi=100, bbox_inches='tight')
plt.show()
print("The Merge was the largest decarbonization event in crypto history.")

---
## 4. Mining Geography & Renewable Energy

Mining geography has shifted dramatically, especially after China's mining ban in May 2021. Understanding where mining occurs determines the carbon intensity of the network.

Key concepts:
- **Stranded energy**: Energy that would otherwise be wasted (flared gas, curtailed renewables)
- **Curtailment mining**: Mining using renewable energy that would be curtailed during low-demand periods
- **Behind-the-meter mining**: Mining co-located with energy generation, avoiding transmission costs

In [ ]:
# Mining geography evolution
mining_distribution = {
    'Pre-Ban (2020)': {
        'China': 0.65, 'USA': 0.08, 'Russia': 0.07,
        'Kazakhstan': 0.06, 'Canada': 0.03, 'Other': 0.11
    },
    'Post-Ban (2022)': {
        'USA': 0.38, 'China': 0.05, 'Russia': 0.12,
        'Kazakhstan': 0.13, 'Canada': 0.07, 'Other': 0.25
    },
    'Current (2024)': {
        'USA': 0.35, 'Russia': 0.12, 'Kazakhstan': 0.10,
        'Canada': 0.10, 'China': 0.05, 'Other': 0.28
    }
}

# Renewable energy adoption in mining
renewable_estimates = {
    '2018': 0.28,  # 28% renewable
    '2019': 0.39,
    '2020': 0.42,
    '2021': 0.46,  # Post China ban, less coal
    '2022': 0.52,
    '2023': 0.55,
    '2024': 0.58,  # Bitcoin Mining Council estimates
}

print("=" * 60)
print("MINING GEOGRAPHY & RENEWABLE ENERGY")
print("=" * 60)

for period, dist in mining_distribution.items():
    print(f"\n{period}:")
    for country, share in sorted(dist.items(), key=lambda x: -x[1]):
        bar = '█' * int(share * 50)
        print(f"  {country:<15} {share:>5.0%} {bar}")

print(f"\nRenewable Energy in Mining:")
for year, pct in renewable_estimates.items():
    bar = '█' * int(pct * 50)
    print(f"  {year}: {pct:.0%} {bar}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mining geography shift
periods = list(mining_distribution.keys())
all_countries = set()
for d in mining_distribution.values():
    all_countries.update(d.keys())
all_countries = sorted(all_countries)

country_colors = plt.cm.Set2(np.linspace(0, 1, len(all_countries)))
x = np.arange(len(periods))
bottom = np.zeros(len(periods))

for j, country in enumerate(all_countries):
    values = [mining_distribution[p].get(country, 0) * 100 for p in periods]
    axes[0].bar(x, values, bottom=bottom, color=country_colors[j], label=country)
    bottom += values

axes[0].set_xticks(x)
axes[0].set_xticklabels(periods, fontsize=9)
axes[0].set_ylabel('Mining Share (%)')
axes[0].set_title('Mining Geography Evolution')
axes[0].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)

# Renewable adoption
years = list(renewable_estimates.keys())
pcts = [v * 100 for v in renewable_estimates.values()]
axes[1].bar(years, pcts, color=plt.cm.Greens(np.array(pcts) / 100))
axes[1].set_ylabel('Renewable Energy (%)')
axes[1].set_title('Renewable Energy in Bitcoin Mining')
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.savefig('/tmp/mining_geography.png', dpi=100, bbox_inches='tight')
plt.show()
print("China's ban shifted mining to North America, increasing renewable share.")

---
## 5. E-Waste from ASIC Hardware

Bitcoin mining generates electronic waste as ASICs become obsolete due to efficiency improvements and halvings.

> **Definition: ASIC (Application-Specific Integrated Circuit)** -- A microchip designed for a single specific purpose. Bitcoin ASICs are designed solely to compute SHA-256 hashes and cannot be repurposed for other tasks, contributing to e-waste when they become unprofitable.

Key factors:
- Average ASIC lifespan: 3-5 years
- Weight: ~12-15 kg per unit
- Contains rare metals, plastics, and toxic materials
- Cannot be repurposed (unlike GPUs)

In [ ]:
def estimate_ewaste(network_hashrate_eh: float, avg_unit_hashrate_th: float = 100,
                    avg_weight_kg: float = 13.5, lifespan_years: float = 4) -> Dict:
    """Estimate e-waste from ASIC mining hardware."""
    total_units = network_hashrate_eh * 1e6 / avg_unit_hashrate_th
    total_weight_tonnes = total_units * avg_weight_kg / 1000
    annual_ewaste_tonnes = total_weight_tonnes / lifespan_years
    
    return {
        'total_units': total_units,
        'total_weight_tonnes': total_weight_tonnes,
        'annual_ewaste_tonnes': annual_ewaste_tonnes,
        'ewaste_per_tx_grams': annual_ewaste_tonnes * 1e6 / (0.3e6 * 365)
    }


# Current e-waste estimate
ewaste = estimate_ewaste(600)

print("=" * 60)
print("ASIC E-WASTE ANALYSIS")
print("=" * 60)

print(f"\nEstimated ASIC units in operation: {ewaste['total_units']:,.0f}")
print(f"Total hardware weight: {ewaste['total_weight_tonnes']:,.0f} tonnes")
print(f"Annual e-waste: {ewaste['annual_ewaste_tonnes']:,.0f} tonnes/year")
print(f"E-waste per transaction: {ewaste['ewaste_per_tx_grams']:.1f} grams")

# ASIC efficiency trend and obsolescence
years = np.arange(2017, 2027)
efficiency_trend = [96, 70, 45, 32, 28, 22, 18, 16, 14, 12]  # J/TH

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Efficiency improvement
axes[0].plot(years, efficiency_trend, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Efficiency (J/TH)')
axes[0].set_title('ASIC Efficiency Improvement')
axes[0].set_ylim(0, 100)

# E-waste projection
hashrate_projections = np.linspace(200, 1000, len(years))
ewaste_projections = [estimate_ewaste(h)['annual_ewaste_tonnes'] for h in hashrate_projections]

axes[1].bar(years, ewaste_projections, color=plt.cm.Oranges(np.linspace(0.3, 0.8, len(years))))
axes[1].set_xlabel('Year')
axes[1].set_ylabel('E-Waste (tonnes/year)')
axes[1].set_title('Projected ASIC E-Waste')

plt.tight_layout()
plt.savefig('/tmp/ewaste_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print("ASIC e-waste grows with hashrate but efficiency gains slow the rate.")

---
## 6. ESG Scoring for Crypto Protocols

> **Definition: ESG (Environmental, Social, Governance)** -- A framework for evaluating a company's or protocol's performance across Environmental impact, Social responsibility, and Governance quality. Increasingly applied to crypto protocols by institutional investors.

Applying ESG to crypto:
- **Environmental**: Energy consumption, carbon footprint, renewable adoption
- **Social**: Financial inclusion, censorship resistance, developer diversity
- **Governance**: Decentralization, upgrade process, token distribution

In [ ]:
@dataclass
class CryptoESG:
    """ESG scoring for a crypto protocol."""
    name: str
    # Environmental (0-10, higher = better)
    energy_efficiency: float
    renewable_pct: float
    carbon_per_tx: float  # Inverse: lower carbon = higher score
    # Social
    accessibility: float
    censorship_resistance: float
    developer_diversity: float
    # Governance
    decentralization: float
    transparency: float
    upgrade_process: float


protocols_esg = [
    CryptoESG("Bitcoin", 2, 6, 2, 9, 10, 7, 9, 8, 6),
    CryptoESG("Ethereum", 9, 7, 9, 8, 8, 9, 8, 9, 8),
    CryptoESG("Solana", 9, 5, 9, 7, 5, 6, 5, 7, 6),
    CryptoESG("Cardano", 9, 6, 9, 7, 7, 6, 7, 8, 7),
    CryptoESG("BNB Chain", 8, 4, 8, 7, 3, 4, 3, 5, 4),
]

def esg_composite(esg: CryptoESG) -> Dict[str, float]:
    """Calculate composite ESG scores."""
    e_score = (esg.energy_efficiency + esg.renewable_pct + esg.carbon_per_tx) / 3
    s_score = (esg.accessibility + esg.censorship_resistance + esg.developer_diversity) / 3
    g_score = (esg.decentralization + esg.transparency + esg.upgrade_process) / 3
    total = (e_score + s_score + g_score) / 3
    return {'E': e_score, 'S': s_score, 'G': g_score, 'Total': total}


print("=" * 60)
print("CRYPTO ESG SCORECARD")
print("=" * 60)

print(f"\n{'Protocol':<12} {'E':>6} {'S':>6} {'G':>6} {'Total':>7} {'Rating':>8}")
print("-" * 48)

for p in protocols_esg:
    scores = esg_composite(p)
    rating = 'A' if scores['Total'] > 7.5 else 'B' if scores['Total'] > 6 else 'C' if scores['Total'] > 4.5 else 'D'
    print(f"{p.name:<12} {scores['E']:>5.1f} {scores['S']:>5.1f} {scores['G']:>5.1f} "
          f"{scores['Total']:>6.1f} {rating:>8}")

# Radar chart comparison
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

categories = ['Energy\nEfficiency', 'Renewable\nAdoption', 'Low\nCarbon',
              'Accessibility', 'Censorship\nResistance', 'Dev\nDiversity',
              'Decentralization', 'Transparency', 'Upgrade\nProcess']
n_cats = len(categories)
angles = np.linspace(0, 2 * np.pi, n_cats, endpoint=False).tolist()
angles += angles[:1]

colors = ['orange', 'blue', 'purple', 'green', 'brown']
for p, color in zip(protocols_esg, colors):
    values = [p.energy_efficiency, p.renewable_pct, p.carbon_per_tx,
              p.accessibility, p.censorship_resistance, p.developer_diversity,
              p.decentralization, p.transparency, p.upgrade_process]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, color=color, label=p.name, alpha=0.7)
    ax.fill(angles, values, alpha=0.05, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=8)
ax.set_ylim(0, 10)
ax.set_title('Crypto ESG Comparison', size=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=8)

plt.tight_layout()
plt.savefig('/tmp/crypto_esg.png', dpi=100, bbox_inches='tight')
plt.show()
print("Bitcoin scores low on E but high on S and G.")
print("Post-Merge Ethereum leads on overall ESG.")

---
## 7. Future Projections

Future energy consumption depends on hashrate growth, hardware efficiency improvements, renewable energy adoption, and potential protocol changes.

In [ ]:
# Future scenario modeling
years_future = np.arange(2024, 2035)

scenarios = {
    'Bull (high growth)': {
        'hashrate_growth': 0.30,      # 30% annual hashrate growth
        'efficiency_improvement': 0.15, # 15% annual efficiency gain
        'renewable_growth': 0.03,      # 3% per year renewable increase
    },
    'Base (moderate)': {
        'hashrate_growth': 0.15,
        'efficiency_improvement': 0.10,
        'renewable_growth': 0.02,
    },
    'Bear (low growth)': {
        'hashrate_growth': 0.05,
        'efficiency_improvement': 0.08,
        'renewable_growth': 0.04,      # More focus on sustainability
    },
}

print("=" * 70)
print("FUTURE ENERGY & CARBON PROJECTIONS")
print("=" * 70)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = {'Bull (high growth)': 'red', 'Base (moderate)': 'blue', 'Bear (low growth)': 'green'}

for scenario_name, params in scenarios.items():
    energy_series = []
    carbon_series = []
    renewable_series = []
    
    hashrate = 600  # Current EH/s
    efficiency = 30  # Current average J/TH
    renewable_pct = 0.58  # Current
    
    for i, year in enumerate(years_future):
        # Update parameters
        if i > 0:
            hashrate *= (1 + params['hashrate_growth'])
            efficiency *= (1 - params['efficiency_improvement'])
            efficiency = max(5, efficiency)  # Physical limit
            renewable_pct = min(0.95, renewable_pct + params['renewable_growth'])
        
        # Calculate energy
        power_gw = hashrate * 1e6 * efficiency / 1e9
        energy_twh = power_gw * 8760 / 1000
        
        # Carbon (accounting for renewable mix)
        fossil_ci = 500  # gCO2/kWh for fossil
        renewable_ci = 20  # gCO2/kWh for renewable
        blended_ci = fossil_ci * (1 - renewable_pct) + renewable_ci * renewable_pct
        carbon_mt = energy_twh * 1e9 * blended_ci / 1e12
        
        energy_series.append(energy_twh)
        carbon_series.append(carbon_mt)
        renewable_series.append(renewable_pct * 100)
    
    color = colors[scenario_name]
    axes[0].plot(years_future, energy_series, color=color, linewidth=2, label=scenario_name)
    axes[1].plot(years_future, carbon_series, color=color, linewidth=2, label=scenario_name)
    axes[2].plot(years_future, renewable_series, color=color, linewidth=2, label=scenario_name)
    
    print(f"\n{scenario_name}:")
    print(f"  2024: {energy_series[0]:.0f} TWh, {carbon_series[0]:.1f} Mt CO2, {renewable_series[0]:.0f}% renewable")
    print(f"  2034: {energy_series[-1]:.0f} TWh, {carbon_series[-1]:.1f} Mt CO2, {renewable_series[-1]:.0f}% renewable")

axes[0].set_xlabel('Year')
axes[0].set_ylabel('Energy (TWh/yr)')
axes[0].set_title('Energy Consumption')
axes[0].legend(fontsize=8)

axes[1].set_xlabel('Year')
axes[1].set_ylabel('CO2 Emissions (Mt/yr)')
axes[1].set_title('Carbon Emissions')
axes[1].legend(fontsize=8)

axes[2].set_xlabel('Year')
axes[2].set_ylabel('Renewable (%)')
axes[2].set_title('Renewable Energy Adoption')
axes[2].legend(fontsize=8)
axes[2].set_ylim(0, 100)

plt.tight_layout()
plt.savefig('/tmp/future_projections.png', dpi=100, bbox_inches='tight')
plt.show()
print("\nKey insight: Efficiency gains and renewable adoption can offset hashrate growth.")
print("Carbon emissions may decrease even as energy consumption increases.")

---
## Exercises

### Exercise 1: Mining Profitability & Carbon Calculator

Build a calculator that determines mining profitability while accounting for carbon offset costs.

**Hints:**
- Revenue = blocks mined * block reward * BTC price
- Energy cost = power consumption * electricity rate
- Carbon cost = emissions * carbon credit price
- Net profit = revenue - energy cost - carbon cost - hardware amortization

In [ ]:
class GreenMiningCalculator:
    """Mining profitability calculator with carbon accounting."""
    
    def __init__(self, hardware: MiningHardware,
                 electricity_rate: float = 0.05,
                 carbon_credit_price: float = 50) -> None:
        """Initialize calculator."""
        self.hardware = hardware
        self.elec_rate = electricity_rate
        self.carbon_price = carbon_credit_price
        # YOUR CODE HERE
    
    def daily_revenue(self, btc_price: float, network_hashrate_eh: float) -> float:
        """Calculate daily mining revenue."""
        # YOUR CODE HERE
        pass
    
    def daily_cost(self, energy_mix: Dict[str, float]) -> Dict[str, float]:
        """Calculate daily costs including carbon."""
        # YOUR CODE HERE
        pass
    
    def breakeven_btc_price(self, network_hashrate_eh: float) -> float:
        """Find breakeven BTC price including carbon costs."""
        # YOUR CODE HERE
        pass

### Exercise 2: Renewable Energy Optimizer

Build an optimizer that determines the optimal mix of renewable and fossil energy for a mining operation to maximize profit while meeting ESG targets.

**Hints:**
- Renewable energy has lower marginal cost but higher capex
- Solar/wind are intermittent -- need battery storage or grid backup
- Carbon regulations may impose costs on fossil usage
- Model curtailment: excess renewable energy can be used for mining

In [ ]:
class RenewableEnergyOptimizer:
    """Optimize energy mix for mining operations."""
    
    def __init__(self, mining_capacity_mw: float = 100) -> None:
        """Initialize optimizer."""
        self.capacity = mining_capacity_mw
        # YOUR CODE HERE
    
    def add_energy_source(self, name: str, capacity_mw: float,
                          cost_per_kwh: float, carbon_intensity: float,
                          availability: float = 1.0) -> None:
        """Add an energy source option."""
        # YOUR CODE HERE
        pass
    
    def optimize(self, carbon_target: float = None) -> Dict:
        """Find optimal energy mix."""
        # YOUR CODE HERE
        pass

### Exercise 3: Blockchain Sustainability Index

Create a comprehensive sustainability index that scores and ranks blockchain protocols on environmental impact.

**Hints:**
- Combine energy per transaction, carbon per transaction, e-waste generation
- Normalize across protocols for fair comparison
- Weight factors based on environmental significance
- Track improvements over time

In [ ]:
class SustainabilityIndex:
    """Blockchain sustainability scoring index."""
    
    def __init__(self) -> None:
        """Initialize index."""
        self.protocols: Dict[str, Dict] = {}
        # YOUR CODE HERE
    
    def add_protocol(self, name: str, energy_per_tx_kwh: float,
                     carbon_per_tx_g: float, ewaste_per_tx_g: float,
                     renewable_pct: float) -> None:
        """Add a protocol to the index."""
        # YOUR CODE HERE
        pass
    
    def score(self, name: str) -> float:
        """Calculate sustainability score (0-100)."""
        # YOUR CODE HERE
        pass
    
    def rank_all(self) -> List[Tuple[str, float]]:
        """Rank all protocols by sustainability."""
        # YOUR CODE HERE
        pass

---
## Summary

### What You Learned
- [x] Estimating Bitcoin energy consumption from hashrate using CBECI methodology
- [x] Calculating carbon footprints using regional energy mix and emission factors
- [x] Quantifying the Ethereum Merge's ~99.95% energy reduction
- [x] Analyzing mining geography shifts and renewable energy adoption trends
- [x] Modeling ASIC e-waste and hardware lifecycle environmental impacts
- [x] Applying ESG scoring frameworks to crypto protocols
- [x] Projecting future energy scenarios under different growth assumptions

### Key Takeaways
1. **Bitcoin's energy use is significant** but context matters -- compare to banking system (~260 TWh/yr)
2. **Carbon intensity depends on location** -- mining in Iceland is ~50x cleaner than Kazakhstan
3. **The Merge proved PoS viability** -- 99.95% energy reduction without sacrificing security
4. **Renewable adoption is increasing** -- now >55% of Bitcoin mining uses sustainable energy
5. **ASIC e-waste is an underreported problem** -- single-purpose hardware cannot be repurposed
6. **Efficiency improvements partially offset hashrate growth** -- but energy may still increase
7. **ESG frameworks are being applied to crypto** -- institutional adoption requires sustainability credentials

### Further Reading
- Cambridge Centre for Alternative Finance. (2024). "CBECI Methodology." cbeci.org
- de Vries, A. (2023). "Bitcoin's Energy Consumption." *Joule*.
- Bitcoin Mining Council. (2024). "BMC Survey Results." bitcoinminingcouncil.com

### Next Steps
- This is the final notebook in the series! Review the complete course:
- [Section 01: Historical Evolution](../sections/01-historical-evolution.md) -- Start from the beginning
- [Section 09: Sustainability](../sections/09-sustainability.md) -- Deep dive into sustainability theory